# Custom-LLM 100M training on Google Colab

This notebook streams Hugging Face datasets, writes tokenized NumPy shards, and trains the 100M model without downloading the full corpora.

[Open this notebook in Colab](https://colab.research.google.com/github/Jaydeep-Nadkarni/Custom-LLM/blob/main/LLM.ipynb)

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime in Colab.'
print(torch.cuda.get_device_name(0))
print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
from pathlib import Path
if not Path('/content/Custom-LLM').exists():
    !git clone https://github.com/Jaydeep-Nadkarni/Custom-LLM.git /content/Custom-LLM
%cd /content/Custom-LLM
%pip install -q -r requirements.txt

## Persist checkpoints in Google Drive

Mount Drive before training so an interrupted Colab session can resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/Custom-LLM-100M')
CHECKPOINT = DRIVE_ROOT / 'checkpoints/latest.pt'
LOG_FILE = DRIVE_ROOT / 'logs/training.log'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

## Stream and tokenize training data

This starts with FineWeb-Edu plus English Wikipedia. The Stack is intentionally not included because it is very large; add it only for a code-focused model. `MAX_TOKENS` controls disk use.

In [ ]:
%cd /content/Custom-LLM
MAX_TOKENS = 200_000_000
!python data/scripts/tokenize_huggingface.py \
  --source HuggingFaceFW/fineweb-edu:default \
  --source wikimedia/wikipedia:20231101.en \
  --max-tokens $MAX_TOKENS \
  --output-dir data/tokenized/100m

In [ ]:
from pathlib import Path
import numpy as np
inputs = sorted(Path('data/tokenized/100m/inputs').glob('*.npy'))
targets = sorted(Path('data/tokenized/100m/targets').glob('*.npy'))
assert inputs and len(inputs) == len(targets)
sequences = sum(len(np.load(path, mmap_mode='r')) for path in inputs)
print(f'shards: {len(inputs):,}')
print(f'sequences: {sequences:,}')
print(f'tokens: {sequences * 512:,}')

## Train the 100M model

The checked-in configuration is `d_model=768`, `10` layers, `12` heads, `512` context, and about `101M` parameters. Start with 20,000 steps; increase `--steps` after confirming GPU memory and loss behavior.

In [ ]:
%cd /content/Custom-LLM
import subprocess
command = [
    'python', '-m', 'training.train',
    '--config', 'configs/100m.yaml',
    '--checkpoint-path', str(CHECKPOINT),
    '--log-file', str(LOG_FILE),
    '--checkpoint-every', '500',
]
if CHECKPOINT.exists():
    command.extend(['--resume-from', str(CHECKPOINT)])
subprocess.run(command, check=True)


## Optional generation

Use the checkpoint with the repository's `TransformerLM` inference code after training.

In [ ]:
%cd /content/Custom-LLM
print('Checkpoint:', CHECKPOINT)
print('Exists:', CHECKPOINT.exists())
if CHECKPOINT.exists():
    print(f'Size: {CHECKPOINT.stat().st_size / 1e9:.2f} GB')